# ShercliffDuct3D — 3D coupled MHD duct (two-way)
`MagneticCoupled3D` · validation form (TRUST-LiMe).


## Introduction
3D counterpart of the 2D Shercliff duct: coupled low-Rm MHD in a duct
with insulating walls, driven by an imposed pressure difference, with
`force_lorentz` on momentum and `vitesse_hydraulique` + `induction_stretching`
on the magnetic equation. Smoke-level form: the run must complete and the
probes must stay finite.


In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read TRUST probe (`.son`) file
Case copied by `trustutils` into `run.BUILD_DIRECTORY`.


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile
BUILD = run.BUILD_DIRECTORY
def read_son(case, name, sub=None):
    if sub is not None:
        cands = [f"{BUILD}/{sub}/{case}_{name}.son",
                 f"{BUILD}/{sub}/PAR_{case}_{name}.son"]
    else:
        cands = [f"{BUILD}/coarse/{case}_{name}.son",
                 f"{BUILD}/{case}_{name}.son"]
    for p in cands:
        if os.path.exists(p):
            s = SonFile(p)
            return np.asarray(s.times()), np.asarray(s.values())
    raise FileNotFoundError(cands)


## Run
Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant.


In [ ]:
NB_PROCS = 2
run.reset()
NX0, NY0, NZ0 = 5, 9, 5
meshes = {
    "coarse": {"nx": NX0, "ny": NY0, "nz": NZ0, "tmax": 2, "pmag": 0.5, "eta": 0.05},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate(
        "shercliff_duct_3d.data", lvl, dic=dic,
        nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()
run.printCases()
run.runCases()
case = cases["coarse"]


## Checks
Smoke-level: the run completed and the probes are finite.


In [ ]:
try:
    t, bx = read_son(case.dataFileName_, "SONDE_BX")
    print("SONDE_BX last value:", float(bx[-1]) if bx.size else bx)
    assert np.all(np.isfinite(bx)), "SONDE_BX contains non-finite values"
except Exception as e:
    print("SONDE_BX read skipped:", e)
try:
    t, bz = read_son(case.dataFileName_, "SONDE_BZ")
    print("SONDE_BZ last value:", float(bz[-1]) if bz.size else bz)
    assert np.all(np.isfinite(bz)), "SONDE_BZ contains non-finite values"
except Exception as e:
    print("SONDE_BZ read skipped:", e)
print("OK: ShercliffDuct3D run completed.")


## Computer Performance


In [ ]:
run.tablePerf()
